# Laplace Approximation: Mevcut PyTorch Modeline Post-hoc Belirsizlik Ekleme

Bu notebook, daha önce deterministik olarak eğitilmiş bir PyTorch regresyon modelini **yeniden full BNN olarak eğitmeden** uncertainty-aware hale getirmeyi gösterir.

Akış:

```text
mevcut PyTorch modeli
      ↓
MAP / deterministik ağırlıklar
      ↓
Laplace approximation
      ↓
θ | D ≈ N(θ_MAP, H^{-1})
      ↓
posterior predictive
      ↓
kapasite senaryoları
      ↓
Pyomo + HiGHS
      ↓
risk-duyarlı kapasite planı
```

Bu örnekte `laplace-torch` kullanılır. Kütüphane full-network, subnetwork ve last-layer Laplace approximation destekler. Öğretim örneğinde **last-layer Laplace** kullanıyoruz; çünkü mevcut bir PyTorch modeline düşük maliyetle post-hoc belirsizlik eklemek için en pragmatik seçeneklerden biridir.

> Kritik ayrım: Laplace Approximation tam Bayesçi posterior değildir. Posterior, MAP çözümü çevresinde yerel Gaussian yaklaşımıyla temsil edilir.


In [ ]:
# Gerekirse:
# %pip install torch laplace-torch numpy pandas matplotlib pyomo highspy

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from laplace import Laplace
import pyomo.environ as pyo

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float32)


## 1. Sentetik üretim kapasitesi verisi

Bir hattın günlük kullanılabilir kapasitesini şu koşullara göre tahmin ettiğimizi düşünelim:

- yük oranı,
- sıcaklık,
- bakım skoru,
- vardiya deneyimi.

Gerçek projede bu veri MES/SCADA/IoT sistemlerinden gelebilir.

Amaç önce deterministik bir sinir ağı eğitmek, sonra bu **mevcut modeli** Laplace approximation ile uncertainty-aware hale getirmektir.


In [ ]:
n = 420

load = np.random.uniform(0.35, 0.98, n)
temperature = np.random.normal(25.0, 5.0, n)
maintenance = np.random.uniform(0.20, 1.00, n)
experience = np.random.uniform(0.10, 1.00, n)

true_mean = (
    82.0
    + 34.0 * np.tanh(2.1 * (load - 0.58))
    - 0.55 * np.abs(temperature - 23.0)
    + 12.0 * maintenance
    + 8.0 * experience
    - 15.0 * (load - 0.84) ** 2
)

noise_sd = 5.5
capacity = true_mean + np.random.normal(0.0, noise_sd, n)

X_raw = np.column_stack(
    [load, temperature, maintenance, experience]
).astype(np.float32)
y_raw = capacity.astype(np.float32)

# Zamansal/operasyonel holdout benzeri basit ayrım
n_train = 320
X_train_raw, X_test_raw = X_raw[:n_train], X_raw[n_train:]
y_train_raw, y_test_raw = y_raw[:n_train], y_raw[n_train:]

X_mean = X_train_raw.mean(axis=0, keepdims=True)
X_std = X_train_raw.std(axis=0, keepdims=True) + 1e-6
y_mean = float(y_train_raw.mean())
y_std = float(y_train_raw.std() + 1e-6)

X_train = torch.tensor((X_train_raw - X_mean) / X_std)
X_test = torch.tensor((X_test_raw - X_mean) / X_std)
y_train = torch.tensor(((y_train_raw - y_mean) / y_std).reshape(-1, 1))
y_test = torch.tensor(((y_test_raw - y_mean) / y_std).reshape(-1, 1))

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True,
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=128,
    shuffle=False,
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 2. Deterministik PyTorch modeli

Bu model normal bir `nn.Module`. Yani burada henüz BNN yok.

\[
\hat y=f_{\theta}(x)
\]

Ağ MSE ile eğitiliyor ve elde edilen ağırlıklar daha sonra Laplace approximation için MAP noktası olarak kullanılıyor.


In [ ]:
class CapacityNet(nn.Module):
    def __init__(self, d=4):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(d, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
        )
        self.output_layer = nn.Linear(16, 1)

    def forward(self, x):
        h = self.feature_extractor(x)
        return self.output_layer(h)


model = CapacityNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
loss_fn = nn.MSELoss()

losses = []

for epoch in range(900):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss) * len(xb)

    losses.append(epoch_loss / len(X_train))

model.eval()

with torch.no_grad():
    test_pred_z = model(X_test).squeeze(-1)
    test_pred = test_pred_z.numpy() * y_std + y_mean

rmse = np.sqrt(np.mean((test_pred - y_test_raw) ** 2))

print("Deterministik test RMSE:", round(float(rmse), 3))

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.show()


## 3. MAP çevresinde Laplace approximation

Laplace yaklaşımının temel formu:

\[
p(\theta\mid D)
\approx
\mathcal N
\left(
\theta_{\text{MAP}},
H^{-1}
\right)
\]

Burada \(H\), negatif log posteriorun MAP çevresindeki curvature/Hessian bilgisidir.

Bu notebookta:

- `subset_of_weights="last_layer"`: yalnız son lineer katman uncertainty-aware,
- `hessian_structure="full"`: son katman için dense curvature yaklaşımı,
- `likelihood="regression"`: Gaussian regresyon likelihood'u.

Bu yöntem **Bayesian Last Layer ile aynı şey değildir**:

- BLL notebookunda son katman için conjugate Bayesian linear regression posterioru analitik hesaplanıyordu.
- Burada posterior, eğitilmiş MAP modelinin çevresindeki curvature kullanılarak **Laplace approximation** ile elde ediliyor.


In [ ]:
# Deterministik modelin normalize edilmiş residual'larından
# homoskedastik observation noise tahmini.
with torch.no_grad():
    train_residual = model(X_train) - y_train
    sigma_noise_z = float(
        torch.sqrt(torch.mean(train_residual ** 2)).clamp_min(1e-3)
    )

print("Normalize sigma_noise tahmini:", round(sigma_noise_z, 4))

la = Laplace(
    model,
    likelihood="regression",
    subset_of_weights="last_layer",
    hessian_structure="full",
    sigma_noise=sigma_noise_z,
    prior_precision=1.0,
    last_layer_name="output_layer",
)

la.fit(train_loader)

# Prior precision'i marginal likelihood ile post-hoc ayarla.
la.optimize_prior_precision(
    pred_type="glm",
    method="marglik",
    n_steps=80,
    lr=0.08,
    prior_structure="scalar",
)

print("Optimize prior precision:", la.prior_precision)


## 4. Posterior predictive belirsizlik

`laplace-torch` regression GLM predictive çağrısı:

\[
\mu_f(x), \quad Var[f(x)\mid D]
\]

üretir.

Observation-level senaryo üretmek için model/function uncertainty'ye gözlem gürültüsünü de ekliyoruz:

\[
Var[Y\mid x,D]
\approx
Var[f(x)\mid D]+\sigma_\epsilon^2.
\]

Bu ayrım önemlidir:

- `function variance` → parametre/model belirsizliği,
- `sigma_noise²` → aleatorik observation noise.


In [ ]:
with torch.no_grad():
    f_mean_z, f_var_z = la(
        X_test,
        pred_type="glm",
        link_approx="probit",
        n_samples=100,
    )

f_mean_z = f_mean_z.squeeze(-1)
f_var_z = f_var_z.squeeze(-1).squeeze(-1)

predictive_var_z = f_var_z + la.sigma_noise ** 2
predictive_sd_z = torch.sqrt(predictive_var_z.clamp_min(1e-10))

mean_pred = f_mean_z.numpy() * y_std + y_mean
sd_pred = predictive_sd_z.numpy() * y_std

lower = mean_pred - 1.645 * sd_pred
upper = mean_pred + 1.645 * sd_pred

coverage90 = np.mean(
    (y_test_raw >= lower) & (y_test_raw <= upper)
)
avg_width90 = np.mean(upper - lower)

laplace_rmse = np.sqrt(np.mean((mean_pred - y_test_raw) ** 2))

print("Laplace mean RMSE:", round(float(laplace_rmse), 3))
print("%90 predictive interval coverage:", round(float(coverage90), 3))
print("Ortalama %90 interval genişliği:", round(float(avg_width90), 3))


In [ ]:
order = np.argsort(mean_pred)

plt.figure(figsize=(10, 4))
plt.plot(y_test_raw[order], label="Gerçek kapasite")
plt.plot(mean_pred[order], label="Laplace predictive mean")
plt.fill_between(
    np.arange(len(order)),
    lower[order],
    upper[order],
    alpha=0.20,
    label="%90 predictive interval",
)
plt.xlabel("Test gözlemleri (mean'e göre sıralı)")
plt.ylabel("Kapasite")
plt.legend()
plt.show()


## 5. Gelecek koşulunda kapasite senaryoları

Planlama noktası:

- yük = 0.90,
- sıcaklık = 31°C,
- bakım skoru = 0.66,
- deneyim = 0.72.

Deterministik model bu nokta için tek kapasite değeri verir.

Laplace approximation ise yaklaşık predictive dağılım üretir. Bu dağılımdan scenario generation yapabiliriz.


In [ ]:
future_raw = np.array(
    [[0.90, 31.0, 0.66, 0.72]],
    dtype=np.float32,
)

future_X = torch.tensor(
    (future_raw - X_mean) / X_std
)

with torch.no_grad():
    deterministic_z = model(future_X).item()

    future_mean_z, future_var_z = la(
        future_X,
        pred_type="glm",
        link_approx="probit",
        n_samples=100,
    )

future_mean_z = float(future_mean_z.item())
future_function_var_z = float(future_var_z.item())
future_total_var_z = (
    future_function_var_z + float(la.sigma_noise.item()) ** 2
)

deterministic_capacity = deterministic_z * y_std + y_mean
laplace_mean_capacity = future_mean_z * y_std + y_mean
laplace_sd_capacity = np.sqrt(future_total_var_z) * y_std

rng = np.random.default_rng(SEED)
capacity_samples = rng.normal(
    laplace_mean_capacity,
    laplace_sd_capacity,
    size=4000,
)
capacity_samples = np.clip(capacity_samples, 0.0, None)

print("Deterministik kapasite:", round(deterministic_capacity, 2))
print("Laplace mean kapasite:", round(laplace_mean_capacity, 2))
print("Laplace predictive SD:", round(laplace_sd_capacity, 2))
print("%05:", round(float(np.quantile(capacity_samples, 0.05)), 2))
print("%95:", round(float(np.quantile(capacity_samples, 0.95)), 2))


## 6. Kapasite rezervasyonu: deterministik karar vs uncertainty-aware karar

Tek hat için şu kararı düşünelim:

\[
x = \text{önceden rezerve edilen kapasite}.
\]

Gerçekte kullanılabilir kapasite \(C_s\) belirsiz.

Scenario bazında teslim edilen üretim:

\[
d_s\le x,
\qquad
d_s\le C_s.
\]

Eksik üretim:

\[
u_s\ge D-d_s.
\]

Amaç:

\[
\min_x
c_x x+
\lambda\frac1S\sum_s u_s.
\]

Bu problem, “mevcut deterministik NN'yi post-hoc uncertainty-aware hale getirmenin” OR açısından neden değerli olabileceğini gösterir.


In [ ]:
S = 600
scenario_idx = rng.choice(
    len(capacity_samples),
    size=S,
    replace=False,
)
scenario_capacity = capacity_samples[scenario_idx]

demand = 120.0
reservation_cost = 1.8
shortage_penalty = 11.0
max_reservation = 145.0

m = pyo.ConcreteModel()
m.S = pyo.RangeSet(0, S - 1)

m.x = pyo.Var(
    domain=pyo.NonNegativeReals,
    bounds=(0, max_reservation),
)
m.delivered = pyo.Var(
    m.S,
    domain=pyo.NonNegativeReals,
)
m.shortage = pyo.Var(
    m.S,
    domain=pyo.NonNegativeReals,
)

m.plan_link = pyo.Constraint(
    m.S,
    rule=lambda M, s: M.delivered[s] <= M.x,
)

m.capacity_link = pyo.Constraint(
    m.S,
    rule=lambda M, s: (
        M.delivered[s] <= float(scenario_capacity[s])
    ),
)

m.shortage_def = pyo.Constraint(
    m.S,
    rule=lambda M, s: (
        M.shortage[s] >= demand - M.delivered[s]
    ),
)

m.obj = pyo.Objective(
    expr=(
        reservation_cost * m.x
        + shortage_penalty
        * (1.0 / S)
        * sum(m.shortage[s] for s in m.S)
    ),
    sense=pyo.minimize,
)

result = pyo.SolverFactory("appsi_highs").solve(m)

laplace_plan = float(pyo.value(m.x))

print("Laplace scenario planı:", round(laplace_plan, 2))
print("Solver:", result.solver.termination_condition)


## 7. Deterministik baseline ve out-of-sample karar testi

Deterministik baseline yalnız modelin tek nokta tahminini kullanır.

Daha sonra her iki kararı **aynı gerçek sentetik kapasite dağılımında** test ediyoruz.

Bu nedenle değerlendirme sadece model metriği değildir; doğrudan downstream karar kalitesidir.


In [ ]:
deterministic_plan = min(
    max(deterministic_capacity, 0.0),
    max_reservation,
)

# Sentetik gerçek data-generating mechanism altında
# aynı gelecek koşulu için bağımsız test.
future_load = future_raw[0, 0]
future_temp = future_raw[0, 1]
future_maint = future_raw[0, 2]
future_exp = future_raw[0, 3]

true_future_mean = (
    82.0
    + 34.0 * np.tanh(2.1 * (future_load - 0.58))
    - 0.55 * abs(future_temp - 23.0)
    + 12.0 * future_maint
    + 8.0 * future_exp
    - 15.0 * (future_load - 0.84) ** 2
)

true_capacity_test = rng.normal(
    true_future_mean,
    noise_sd,
    size=20000,
)
true_capacity_test = np.clip(true_capacity_test, 0.0, None)


def realized_cost(plan, available_capacity):
    delivered = np.minimum(plan, available_capacity)
    shortage = np.maximum(demand - delivered, 0.0)

    return (
        reservation_cost * plan
        + shortage_penalty * shortage
    )


def empirical_cvar(costs, alpha=0.95):
    q = np.quantile(costs, alpha)
    tail = costs[costs >= q]
    return float(tail.mean())


cost_det = realized_cost(
    deterministic_plan,
    true_capacity_test,
)
cost_laplace = realized_cost(
    laplace_plan,
    true_capacity_test,
)

comparison = pd.DataFrame(
    {
        "Yöntem": [
            "Deterministik NN",
            "NN + Laplace + stochastic optimization",
        ],
        "Plan": [
            deterministic_plan,
            laplace_plan,
        ],
        "Beklenen maliyet": [
            cost_det.mean(),
            cost_laplace.mean(),
        ],
        "CVaR95": [
            empirical_cvar(cost_det),
            empirical_cvar(cost_laplace),
        ],
        "Shortage olasılığı": [
            np.mean(
                np.minimum(
                    deterministic_plan,
                    true_capacity_test,
                )
                < demand
            ),
            np.mean(
                np.minimum(
                    laplace_plan,
                    true_capacity_test,
                )
                < demand
            ),
        ],
    }
)

comparison


## 8. Last-layer, full-network ve subnetwork Laplace

`laplace-torch` yalnız last-layer yaklaşımıyla sınırlı değildir.

### Last-layer Laplace

```python
Laplace(
    model,
    "regression",
    subset_of_weights="last_layer",
    hessian_structure="full",
)
```

Avantaj:

- ucuz,
- mevcut modelde hızlı uygulanır,
- posterior boyutu küçüktür.

Sınırlama:

- feature extractor/backbone belirsizliği hesaba katılmaz.

### Full-network Laplace

```python
Laplace(
    model,
    "regression",
    subset_of_weights="all",
    hessian_structure="diag",
)
```

Daha fazla parametre uncertainty-aware olur, fakat curvature hesaplama ve posterior approximation maliyeti yükselir.

### Subnetwork Laplace

Modelin yalnız seçili parametre altkümesi uncertainty-aware yapılabilir.

Bu, özellikle büyük endüstriyel ağlarda “nerede belirsizlik gerekli?” sorusuna pragmatik cevap verebilir.


## 9. Laplace ne zaman mantıklı?

Güçlü kullanım senaryosu:

```text
zaten eğitilmiş PyTorch modeli var
        +
full BNN yeniden eğitmek pahalı
        +
uncertainty downstream kararda gerekli
```

Örneğin:

- kapasite planlama,
- predictive maintenance,
- kalite risk analizi,
- surrogate optimization,
- dijital ikiz,
- enerji planlama.

Ancak Laplace approximation **yerel Gaussian yaklaşımıdır**. Posterior güçlü biçimde multimodal veya MAP çevresi Gaussian'dan çok uzaksa approximation kalitesi düşebilir.

Bu nedenle gerçek projede en az şu baseline'larla kıyaslanmalıdır:

- deterministic NN,
- Deep Ensemble,
- Bayesian Last Layer,
- full/variational BNN,
- Gaussian Process (uygun veri rejiminde).

Asıl başarı metriği yalnız RMSE değil:

\[
\text{calibration}
\rightarrow
\text{constraint violation}
\rightarrow
\text{expected cost}
\rightarrow
\text{CVaR}.
\]
